# 🗣️ PLN — Processamento de Linguagem Natural com Machine Learning
## Um Guia Completo e Bem Estruturado

---

### 📚 Sumário

| Parte | Tópico | Descrição |
|-------|--------|-----------|
| **1** | Introdução ao PLN | O que é, exemplos do dia a dia, desafios e pipeline geral |
| **2** | Setup e Instalações | Dependências, imports e downloads do NLTK |
| **3** | Tokenização | Dividir texto em unidades menores (palavras, sentenças) e normalizar |
| **4** | Stopwords | Remoção de palavras irrelevantes — e quando **não** removê-las |
| **5** | Bag of Words (BoW) | Representação numérica de texto por contagem de frequências |
| **6** | TF-IDF | Pesos inteligentes que destacam palavras realmente relevantes |
| **7** | BoW vs TF-IDF | Comparação prática lado a lado com o mesmo corpus |
| **8** | Classificação de Sentimentos | Naive Bayes e Logistic Regression aplicados a reviews |
| **9** | Word Embeddings | Vetores semânticos densos (Word2Vec) — conceito e demo |
| **10** | Conclusão | Resumo do pipeline, checklist e próximos passos |

> **Público-alvo:** Estudantes de ADS com base em Python/SQL, começando em IA/PLN.  
> Cada seção responde três perguntas: **O que é?** → **Por que importa?** → **Como usar na prática?**


---
# PARTE 1: Introdução ao PLN

## 🚀 O que é PLN (Processamento de Linguagem Natural)?

**PLN** é a área da Inteligência Artificial que permite computadores **entenderem, interpretarem e gerarem texto** escrito ou falado por seres humanos. Ele une três campos:

- **Linguística** — como as línguas funcionam (gramática, semântica, pragmática)
- **Ciência da Computação** — algoritmos, estruturas de dados e otimização
- **Inteligência Artificial** — aprendizado de máquina, reconhecimento de padrões

### Exemplos de PLN no seu dia a dia
| Situação | O que o PLN faz por trás |
|----------|------------------------|
| 🔍 Google: "melhor padaria perto de mim" | Interpreta a intenção e localização |
| 📱 WhatsApp sugere "Estou chegando" | Prevê respostas com base no contexto |
| 🎤 Alexa/Siri entendem voz | Converte fala → texto → ação |
| 💬 ChatGPT responde perguntas | Gera texto coerente a partir de contexto |
| 📧 Gmail filtra spam | Classifica e-mails pelo conteúdo |

### Desafios do PLN 🤔

- ❓ **Ambiguidade:** "banco" = instituição financeira ou assento? Depende do contexto!
- 😊 **Ironia/sarcasmo:** "Que produto *maravilhoso*" pode ser negativo.
- 🌐 **Negação:** "não é ruim" → na verdade é positivo, mas contém "ruim".
- 🗨️ **Variações:** "adorei", "amei", "top", "show" transmitem a mesma ideia.

### Pipeline de PLN (Fluxo Geral) 🔀

Este notebook segue **exatamente esse fluxo**, passo a passo:

```
Texto Bruto  (string crua)
    ↓ 🔤 Tokenização  (dividir em palavras)          ← PARTE 3
Tokens
    ↓ 🔡 Normalização  (tudo minúsculo)               ← PARTE 3
Tokens normalizados
    ↓ ✂️ Stopwords  (remover palavras genéricas)       ← PARTE 4
Texto limpo
    ↓ 📊 Vetorização  (converter texto em números)     ← PARTES 5, 6 e 7
Matriz numérica
    ↓ 🤖 Modelo  (classificar, agrupar, predizer)     ← PARTE 8
Predição
```

> 👉 **Conexão com a próxima parte:** Antes de começar a processar texto, precisamos instalar e importar as bibliotecas necessárias. Vamos ao **Setup**!


---
# PARTE 2: Setup e Instalações

## 🛠️ Bibliotecas Principais

| Biblioteca | Para que usamos neste notebook |
|------------|-------------------------------|
| **NLTK** | Tokenização (`word_tokenize`, `sent_tokenize`), lista de stopwords |
| **Scikit-learn** | `CountVectorizer` (BoW), `TfidfVectorizer`, `MultinomialNB`, `LogisticRegression` |
| **Pandas** | Visualizar matrizes BoW/TF-IDF como tabelas legíveis (`DataFrame`) |
| **NumPy** | Operações numéricas com vetores e matrizes |
| **Gensim** | Treinar Word2Vec (Word Embeddings) |

> Execute as duas células abaixo **apenas uma vez** no seu ambiente (Colab/Jupyter).


In [ ]:
# =============================================
# INSTALAÇÃO DE DEPENDÊNCIAS # Pyhton 3.13.11
# =============================================
# Por que instalar?
#   Essas bibliotecas NÃO vêm por padrão no Python.
#   Execute esta célula apenas uma vez; depois pode comentá-la.

!pip install nltk scikit-learn pandas numpy gensim


In [2]:
# =============================================
# IMPORTAÇÕES E DOWNLOADS NLTK # Pyhton 3.13.11
# =============================================
# Centralizamos todos os imports aqui no início.
# Assim, se faltar alguma biblioteca, o erro aparece logo (fail fast).

import nltk                                   # Toolkit de PLN
import numpy as np                             # Operações numéricas
import pandas as pd                            # DataFrames para visualização
from sklearn.feature_extraction.text import (  # Vetorizadores
    CountVectorizer,
    TfidfVectorizer,
)

print("✅ Importações básicas concluídas!")

# Baixar recursos essenciais do NLTK (executar apenas 1 vez)
# - punkt_tab  → modelo de tokenização (dividir texto em palavras/sentenças)
# - stopwords  → lista de palavras comuns por idioma (pt, en, etc.)
# - wordnet    → base léxica usada para sinônimos e lemas
for recurso in ['punkt_tab', 'stopwords', 'wordnet']:
    nltk.download(recurso, quiet=True)  # quiet=True evita log excessivo

print("✅ Recursos NLTK baixados!")
print("\n👉 Tudo pronto! Próximo passo: Tokenização — o primeiro passo real do pipeline.")


✅ Importações básicas concluídas!
✅ Recursos NLTK baixados!

👉 Tudo pronto! Próximo passo: Tokenização — o primeiro passo real do pipeline.


---
# PARTE 3: Tokenização em Profundidade

## ✂️ O que é Tokenização?

**Definição:** Tokenização é o processo de dividir um texto cru em unidades menores chamadas **tokens**.  
Um token pode ser uma palavra, número, pontuação ou até emoji.

### Por que é importante no pipeline?

É o **primeiro passo obrigatório**. Sem tokenização, o computador enxerga o texto como uma única string, sem saber onde uma palavra termina e outra começa:

```
❌ SEM tokenização:
   Computador vê: "O Brasil ganhou! Que jogo."
   → Uma string só. Como contar palavras? Como buscar padrões?

✅ COM tokenização:
   Computador vê: ["O", "Brasil", "ganhou", "!", "Que", "jogo", "."]
   → Cada elemento é analisável individualmente.
```

### Tipos de Tokenização

| Tipo | O que faz | Quando usar |
|------|----------|-------------|
| **Por palavras** | Separa cada palavra e pontuação | Maioria dos casos de PLN |
| **Por sentenças** | Separa frases completas | Resumo automático, análise por frase |
| **Por caracteres** | Separa letra por letra | Idiomas sem espaço (chinês), subpalavras |

### O que vamos fazer na célula abaixo:
1. **Tokenizar por palavras** — com `word_tokenize` (NLTK)
2. **Tokenizar por sentenças** — com `sent_tokenize` (NLTK)
3. **Normalizar com `.lower()`** — para que "Python", "PYTHON" e "python" sejam iguais

> 💡 O `word_tokenize` do NLTK usa regras linguísticas (não é um simples `.split()`):
> separa pontuação, trata contrações e preserva padrões como "5-0" ou "$499.99".


In [3]:
# ==============================================================================
# PROCESSO DE TOKENIZAÇÃO E NORMALIZAÇÃO (PIPELINE INICIAL de PLN)
# Objetivo: Transformar strings brutas em unidades atômicas processáveis (Tokens).
# ==============================================================================

from nltk.tokenize import word_tokenize, sent_tokenize

# --- SEÇÃO 1: TOKENIZAÇÃO POR PALAVRAS (Word Tokenization) ---
# Diferente de um simples .split(" "), o word_tokenize utiliza modelos treinados 
# para separar pontuações de palavras e manter estruturas complexas (ex: e-mails).

exemplos = [
    ("Esportivo", "O Brasil ganhou 5-0! Que jogo incrível :)"),
    ("Preço",     "Comprei um celular. O preço era $499.99!"),
    ("Contato",   "E-mail: contato@empresa.com.br - Telefone: (11) 98765-4321")
]

print("=" * 70)
print("🔍 EXPLORANDO A INTELIGÊNCIA DA TOKENIZAÇÃO")
print("=" * 70)

for rotulo, texto in exemplos:
    # O parâmetro language='portuguese' é vital para tratar clíticos e 
    # abreviações específicas da nossa língua.
    tokens = word_tokenize(texto, language='portuguese')
    
    print(f"\n📝 Contexto {rotulo}: {texto}")
    print(f"👉 Tokens Gerados: {tokens}")
    # Nota técnica: Observe como ':)' ou '$499.99' são tratados como unidades.
    print(f"📊 Densidade Léxica: {len(tokens)} unidades identificadas.")

# --- SEÇÃO 2: TOKENIZAÇÃO POR SENTENÇAS (Sentence Segmentation) ---
# Essencial para análises que dependem do contexto da frase, como Resumo 
# Automático ou Tradução, onde a ordem das sentenças importa.

paragrafo = "O Brasil ganhou! A vitória foi inesperada. Que jogo emocionante."
# O sent_tokenize identifica as fronteiras de sentenças através de sinais 
# de pontuação terminal (. ! ?) e regras de capitalização.
sentencas = sent_tokenize(paragrafo, language='portuguese')

print("\n" + "=" * 70)
print("✂️ SEGMENTAÇÃO DE SENTENÇAS (SENTENCE TOKENIZATION)")
print("=" * 70)
for i, sentenca in enumerate(sentencas, 1):
    print(f"📍 Sentença {i}: \"{sentenca}\"")

# --- SEÇÃO 3: NORMALIZAÇÃO DE CASE (Lowercasing) ---
# Este é um passo de redução de dimensionalidade. Para a máquina, 'A' e 'a' 
# possuem códigos ASCII diferentes. Normalizar evita que o modelo aprenda 
# a mesma palavra múltiplas vezes por causa da posição na frase.

texto_caotico = "Python é Linguagem. PYTHON É RÁPIDO. python é popular."

# Sem normalização: O vocabulário terá 3 entradas para a mesma tecnologia.
tokens_brutos = word_tokenize(texto_caotico, language='portuguese')
# Com normalização: Consolidação de tokens idênticos.
tokens_limpos = word_tokenize(texto_caotico.lower(), language='portuguese')

print("\n" + "=" * 70)
print("🔡 NORMALIZAÇÃO LÉXICA (.lower())")
print("=" * 70)
print(f"❌ Vocabulário Bruto: {set(tokens_brutos)}") # set() mostra palavras únicas
print(f"✅ Vocabulário Normalizado: {set(tokens_limpos)}")
print("\n💡 Insight: A normalização reduz o 'ruído' e foca na essência semântica.")

🔍 EXPLORANDO A INTELIGÊNCIA DA TOKENIZAÇÃO

📝 Contexto Esportivo: O Brasil ganhou 5-0! Que jogo incrível :)
👉 Tokens Gerados: ['O', 'Brasil', 'ganhou', '5-0', '!', 'Que', 'jogo', 'incrível', ':', ')']
📊 Densidade Léxica: 10 unidades identificadas.

📝 Contexto Preço: Comprei um celular. O preço era $499.99!
👉 Tokens Gerados: ['Comprei', 'um', 'celular', '.', 'O', 'preço', 'era', '$', '499.99', '!']
📊 Densidade Léxica: 10 unidades identificadas.

📝 Contexto Contato: E-mail: contato@empresa.com.br - Telefone: (11) 98765-4321
👉 Tokens Gerados: ['E-mail', ':', 'contato', '@', 'empresa.com.br', '-', 'Telefone', ':', '(', '11', ')', '98765-4321']
📊 Densidade Léxica: 12 unidades identificadas.

✂️ SEGMENTAÇÃO DE SENTENÇAS (SENTENCE TOKENIZATION)
📍 Sentença 1: "O Brasil ganhou!"
📍 Sentença 2: "A vitória foi inesperada."
📍 Sentença 3: "Que jogo emocionante."

🔡 NORMALIZAÇÃO LÉXICA (.lower())
❌ Vocabulário Bruto: {'É', 'é', 'popular', 'RÁPIDO', 'PYTHON', 'Python', 'Linguagem', '.', 'python'}
✅ Vo

---
# PARTE 4: Remoção de Stopwords

## 🚫 O que são Stopwords?

**Definição:** Stopwords são palavras extremamente comuns em um idioma que, sozinhas, geralmente **não carregam significado útil** para a análise.  
Exemplos em português: "o", "a", "de", "para", "em", "é", "e", "ou".

```
Frase original:       "O gato de estimação é muito bom"
Stopwords removidas:   O, de, é, muito
Resultado:             "gato estimação bom" ← só ficou o que importa!
```

### Por que remover stopwords no pipeline?

Porque na etapa seguinte (vetorização), cada palavra vira uma **coluna na matriz**.  
Se mantivermos stopwords, a matriz fica cheia de colunas inúteis que:
- 🎯 **Adicionam ruído** — dilui o sinal das palavras que realmente importam
- 📉 **Inflam a dimensionalidade** — mais colunas = mais memória e tempo
- ⚡ **Retardam o processamento** — o modelo gasta esforço com dados irrelevantes

### ⚠️ CUIDADO CRÍTICO: Stopwords × Análise de Sentimentos

Nem **sempre** remover stopwords é uma boa ideia! Palavras como **"não"**, **"mas"**, **"nem"**, **"nunca"** são classificadas como stopwords, mas carregam **significado crucial** em análise de sentimentos:

| Frase original | Sem stopwords | Sentimento real | Sentimento detectado |
|---------------|---------------|-----------------|---------------------|
| "**não** é bom" | "bom" | ❌ Negativo | ✅ Positivo (ERRO!) |
| "bom **mas** caro" | "bom caro" | ⚠️ Misto | ✅ Positivo (ERRO!) |
| "**nem** ruim **nem** bom" | "ruim bom" | ⚠️ Neutro | ❓ Confuso |
| "**nunca** mais compro" | "compro" | ❌ Negativo | ✅ Neutro/Positivo (ERRO!) |

> **Dica prática:** Em tarefas de sentimento, crie uma **lista customizada** que
> **preserva negações** ("não", "nem", "nunca") e **conectivos de contraste** ("mas", "porém").
> Vamos demonstrar essa técnica no código abaixo.


In [7]:
# ==============================================================================
# BLOCO 1: EXPLORAÇÃO E AUDITORIA DO CORPUS DE STOPWORDS
# Objetivo: Carregar a base léxica do NLTK e verificar a presença de termos
# que possuem alta carga semântica para análise de sentimento.
# ==============================================================================

from nltk.corpus import stopwords

# Armazenamos em um set() para garantir busca de alta performance O(1).
# Em grandes volumes de texto, o set é drasticamente mais rápido que uma lista.
stopwords_portugues = set(stopwords.words('portuguese'))

print("=" * 70)
print("📚 AUDITORIA DE STOPWORDS - NLTK PORTUGUÊS")
print("=" * 70)
print(f"\nTotal de termos na lista padrão: {len(stopwords_portugues)}")
print(f"Amostra inicial: {sorted(list(stopwords_portugues))[:15]}")

# Verificação de Negações e Contrastes:
# Muitas vezes, palavras fundamentais para o sentido da frase são descartadas 
# como ruído. Abaixo, realizamos uma auditoria manual nesses termos.
print("\n⚠️ VERIFICAÇÃO DE PERDA DE SINAL SEMÂNTICO:")
palavras_criticas = ['não', 'mas', 'nem', 'nunca', 'nada', 'sem']

for palavra in palavras_criticas:
    esta_na_lista = palavra in stopwords_portugues
    emoji = "🚨" if esta_na_lista else "✅"
    # Se estiver na lista, o modelo ficará "cego" para esta negação/contraste.
    status = "SERÁ REMOVIDA (Risco de erro)" if esta_na_lista else "PRESERVADA (Seguro)"
    print(f"   {emoji} '{palavra.ljust(6)}': {status}")

print("\n💡 Conclusão para a IC: O descarte padrão pode 'cegar' o modelo para negações.")

📚 AUDITORIA DE STOPWORDS - NLTK PORTUGUÊS

Total de termos na lista padrão: 207
Amostra inicial: ['a', 'ao', 'aos', 'aquela', 'aquelas', 'aquele', 'aqueles', 'aquilo', 'as', 'até', 'com', 'como', 'da', 'das', 'de']

⚠️ VERIFICAÇÃO DE PERDA DE SINAL SEMÂNTICO:
   🚨 'não   ': SERÁ REMOVIDA (Risco de erro)
   🚨 'mas   ': SERÁ REMOVIDA (Risco de erro)
   🚨 'nem   ': SERÁ REMOVIDA (Risco de erro)
   ✅ 'nunca ': PRESERVADA (Seguro)
   ✅ 'nada  ': PRESERVADA (Seguro)
   🚨 'sem   ': SERÁ REMOVIDA (Risco de erro)

💡 Conclusão para a IC: O descarte padrão pode 'cegar' o modelo para negações.


In [10]:
# ==============================================================================
# BLOCO 2: IMPACTO DA REMOÇÃO DE STOPWORDS NA POLARIDADE (INVERSÃO SEMÂNTICA)
# Objetivo: Evidenciar o risco de 'cegueira algorítmica' ao remover termos que, 
# apesar de frequentes, são os principais carregadores de sentido negativo.
# ==============================================================================

from nltk.tokenize import word_tokenize

# Amostras de teste selecionadas estrategicamente para desafiar o pipeline.
# Note que frases negativas dependem quase inteiramente de "Stopwords" (não, nunca).
lista_reviews = [
    "Adorei o produto! É de ótima qualidade.",            # Sentimento: Positivo
    "Não gostei. Produto de péssima qualidade.",          # Sentimento: Negativo (Crucial: 'Não')
    "É ok. Preço bom mas qualidade ruim.",                # Sentimento: Misto (Crucial: 'mas')
    "Nunca mais compro. Nem funciona direito."            # Sentimento: Negativo (Crucial: 'Nunca'/'Nem')
]

print("=" * 70)
print("📉 SIMULAÇÃO DE PERDA DE CONTEXTO E INVERSÃO DE POLARIDADE")
print("=" * 70)

# Iteração sobre o corpus de teste para avaliar o pipeline de pré-processamento
for i, review in enumerate(lista_reviews, 1):
    
    # PASSO 1: TOKENIZAÇÃO E NORMALIZAÇÃO (Case Folding)
    # Convertemos para minúsculo para garantir que 'Não' e 'não' sejam o mesmo token.
    tokens_brutos = word_tokenize(review.lower(), language='portuguese')

    # PASSO 2: FILTRAGEM DE RUÍDO (Stopwords e Caracteres Especiais)
    # O perigo acadêmico aqui é a 'Filtragem Indiscriminada'.
    # A estrutura .isalpha() elimina pontuações (como '!' ou ':)') e números.
    # O 'if token not in stopwords_portugues' remove os termos da lista padrão do NLTK.
    tokens_filtrados = [
        token for token in tokens_brutos 
        if token not in stopwords_portugues and token.isalpha()
    ]

    print(f"\n📝 Review {i} (Original): \"{review}\"")
    print(f"👉 Tokens Finais após 'limpeza': {tokens_filtrados}")
    
    # NOTA PARA A IC: 
    # Analise a Review 2: Após a remoção, sobra ['gostei', 'péssima', 'qualidade']. 
    # Sem a negação 'não', um modelo de Machine Learning pode interpretar 
    # erroneamente 'gostei' como um sinal positivo forte, ignorando a insatisfação.

📉 SIMULAÇÃO DE PERDA DE CONTEXTO E INVERSÃO DE POLARIDADE

📝 Review 1 (Original): "Adorei o produto! É de ótima qualidade."
👉 Tokens Finais após 'limpeza': ['adorei', 'produto', 'ótima', 'qualidade']

📝 Review 2 (Original): "Não gostei. Produto de péssima qualidade."
👉 Tokens Finais após 'limpeza': ['gostei', 'produto', 'péssima', 'qualidade']

📝 Review 3 (Original): "É ok. Preço bom mas qualidade ruim."
👉 Tokens Finais após 'limpeza': ['ok', 'preço', 'bom', 'qualidade', 'ruim']

📝 Review 4 (Original): "Nunca mais compro. Nem funciona direito."
👉 Tokens Finais após 'limpeza': ['nunca', 'compro', 'funciona', 'direito']


In [11]:
# ==============================================================================
# BLOCO 3: REGENERAÇÃO SEMÂNTICA VIA OPERAÇÃO DE CONJUNTOS (Set Difference)
# Objetivo: Implementar uma solução técnica que concilie a redução de ruído 
# (limpeza) com a preservação de indicadores de polaridade (sinal).
# ==============================================================================

# Definição do 'Safe Set' (Conjunto de Segurança):
# Selecionamos manualmente as stopwords que possuem valor semântico crítico.
# Estas palavras são os "modificadores de sentido" da língua portuguesa.
palavras_a_preservar = {'não', 'mas', 'nem', 'nunca', 'nada', 'sem', 'porém', 'contudo'}

# TÉCNICA: DIFERENÇA DE CONJUNTOS (A - B)
# Em Python, o operador '-' em objetos do tipo 'set' executa uma operação 
# matemática de conjuntos. O resultado é um novo conjunto contendo todos 
# os elementos de 'A' que NÃO estão presentes em 'B'.
# Isso garante que as palavras preservadas NUNCA sejam filtradas pelo pipeline.
stopwords_customizadas = stopwords_portugues - palavras_a_preservar

print("=" * 70)
print("🛠️ OTIMIZAÇÃO TÉCNICA: CRIAÇÃO DA LISTA CUSTOMIZADA")
print("=" * 70)
print(f"\nVolume original (NLTK):  {len(stopwords_portugues)} palavras")
print(f"Volume otimizado (IC):   {len(stopwords_customizadas)} palavras")
print(f"Indicadores resgatados:  {sorted(palavras_a_preservar)}")

# ---- TESTE DE VALIDAÇÃO DO PIPELINE OTIMIZADO ----

review_teste = "Não gostei. Produto de péssima qualidade."
# Normalização e Tokenização padrão
tokens = word_tokenize(review_teste.lower(), language='portuguese')

# Comparativo: List Comprehension com filtros distintos
# 1. Filtro Genérico: Perde a negação 'não'.
tokens_filtro_padrao = [t for t in tokens if t not in stopwords_portugues and t.isalpha()]

# 2. Filtro Customizado: Mantém a negação, essencial para modelos de sentimento.
tokens_filtro_custom = [t for t in tokens if t not in stopwords_customizadas and t.isalpha()]

print(f"\n🔍 AVALIAÇÃO DE RESULTADO (TESTE DE ESTRESSE):")
print(f"Frase original: \"{review_teste}\"")
print(f"❌ Saída Padrão (Ruído):     {tokens_filtro_padrao}")
print(f"✅ Saída Customizada (Sinal): {tokens_filtro_custom}  <-- 'não' PRESERVADO!")

print("\n📈 CONCLUSÃO TÉCNICA:")
print("A aplicação da Teoria dos Conjuntos permitiu o 'Fine-tuning' do dataset.")
print("Com as negações preservadas, o classificador (Naive Bayes ou Logistic Reg.)")
print("será capaz de distinguir a polaridade negativa da positiva corretamente.")

print("\n👉 Próximo estágio do Pipeline: Vetorização (Feature Engineering).")

🛠️ OTIMIZAÇÃO TÉCNICA: CRIAÇÃO DA LISTA CUSTOMIZADA

Volume original (NLTK):  207 palavras
Volume otimizado (IC):   203 palavras
Indicadores resgatados:  ['contudo', 'mas', 'nada', 'nem', 'nunca', 'não', 'porém', 'sem']

🔍 AVALIAÇÃO DE RESULTADO (TESTE DE ESTRESSE):
Frase original: "Não gostei. Produto de péssima qualidade."
❌ Saída Padrão (Ruído):     ['gostei', 'produto', 'péssima', 'qualidade']
✅ Saída Customizada (Sinal): ['não', 'gostei', 'produto', 'péssima', 'qualidade']  <-- 'não' PRESERVADO!

📈 CONCLUSÃO TÉCNICA:
A aplicação da Teoria dos Conjuntos permitiu o 'Fine-tuning' do dataset.
Com as negações preservadas, o classificador (Naive Bayes ou Logistic Reg.)
será capaz de distinguir a polaridade negativa da positiva corretamente.

👉 Próximo estágio do Pipeline: Vetorização (Feature Engineering).


---
# PARTE 5: Bag of Words (BoW)

## 👜 O que é Bag of Words?

**Definição:** Bag-of-Words é uma forma de representar texto como **números**, contando quantas vezes cada palavra aparece em cada documento. Ignora completamente a **ordem** das palavras.

### Por que importa no pipeline?

Até aqui temos tokens limpos (palavras sem stopwords). Mas modelos de Machine Learning **não entendem texto** — eles precisam de **números**. BoW é a forma mais simples e direta de fazer essa conversão.

### Analogia 🎒

Imagine jogar todas as palavras de uma frase dentro de uma mochila. Não importa a ordem — só importa **quantas de cada tipo** tem lá dentro:

```
Frase 1: "O gato subiu no telhado"  →  {gato: 1, subiu: 1, telhado: 1}
Frase 2: "No telhado o gato subiu"  →  {gato: 1, subiu: 1, telhado: 1}
⚠️ MESMA representação! BoW não se importa com ordem.
```

### Como funciona (3 passos)

```
PASSO 1 — Vocabulário: listar todas as palavras únicas do corpus
  Docs: "python é bom", "python é rápido"
  Vocab: {bom, é, python, rápido}

PASSO 2 — Contagem: para cada doc, contar frequência de cada palavra
  Doc1: bom(1) é(1) python(1) rápido(0) → [1, 1, 1, 0]
  Doc2: bom(0) é(1) python(1) rápido(1) → [0, 1, 1, 1]

PASSO 3 — Montar a Matriz BoW
         bom  é  python  rápido
  Doc1    1   1    1       0
  Doc2    0   1    1       1
```

### Propriedades

| ✅ Vantagens | ❌ Limitações |
|-------------|---------------|
| Simples e rápido | Perde ordem das palavras |
| Transparente (fácil de entender) | Matriz esparsa (muitos zeros) |
| Funciona bem para classificação | Não captura semântica |

> Na célula abaixo, usamos o `CountVectorizer` do scikit-learn, que faz os 3 passos automaticamente.


In [12]:
# ==============================================================================
# BLOCO 4: VETORIZAÇÃO - MODELO BAG OF WORDS (SACO DE PALAVRAS)
# Objetivo: Mapear o vocabulário do corpus e converter documentos de texto em 
# vetores numéricos baseados na frequência de ocorrência (Contagem).
# ==============================================================================

from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

# --- CURIOSIDADE TÉCNICA 1 ---
# Por que 'Bag of Words'? O termo reflete que o modelo descarta a sintaxe e a 
# ordem das palavras. Para o BoW, "O gato caça o rato" e "O rato caça o gato" 
# geram vetores IDENTICOS, pois as frequências são as mesmas.
# ----------------------------

print("=" * 70)
print("📊 ENGENHARIA DE ATRIBUTOS: CONTAGEM DE FREQUÊNCIAS (BoW)")
print("=" * 70)

# Corpus de exemplo: 5 documentos (reviews) com sentimentos variados.
reviews_produtos = [
    "Som excelente, muito bom, recomendo!",
    "Produto ruim, som horrível, não recomendo",
    "Bom custo-benefício, som bom",
    "Qualidade excelente, adorei",
    "Horrível, pior compra, muito ruim",
]
rotulos_sentimento = ["Positivo", "Negativo", "Positivo", "Positivo", "Negativo"]

# Instanciação do Vetorizador
# O parâmetro lowercase=True é o padrão e realiza o Case Folding automaticamente.
vetorizador_bow = CountVectorizer(lowercase=True)

# --- CURIOSIDADE TÉCNICA 2 ---
# O fit_transform retorna uma 'Sparse Matrix' (Matriz Esparsa). 
# Em NLP real, o vocabulário pode ter 50.000 palavras, mas uma review só usa 10.
# Armazenar os zeros seria desperdício de RAM. O scikit-learn armazena apenas
# as posições que NÃO são zero (formato CSR - Compressed Sparse Row).
# ----------------------------

# PASSO 1: fit() -> Escaneia o corpus e constrói o índice do vocabulário.
# PASSO 2: transform() -> Codifica cada texto em um vetor de frequências.
matriz_bow = vetorizador_bow.fit_transform(reviews_produtos)

# Recuperação dos 'Features Names' (as palavras que viraram colunas)
vocabulario_bow = vetorizador_bow.get_feature_names_out()

print(f"\n✨ Vocabulário Mapeado ({len(vocabulario_bow)} palavras únicas):")
print(f"👉 {list(vocabulario_bow)}")

# --- CONVERSÃO PARA DATAFRAME (VISUALIZAÇÃO DIDÁTICA) ---
# Em produção, evitamos usar .toarray() em matrizes gigantes para não estourar a RAM.
tabela_bow = pd.DataFrame(
    matriz_bow.toarray(), 
    columns=vocabulario_bow, 
    index=[f"Review {i+1} ({r})" for i, r in enumerate(rotulos_sentimento)]
)

print("\n📊 REPRESENTAÇÃO MATRICIAL (VETORES DE FREQUÊNCIA):")
print(tabela_bow)

# --- ANÁLISE ESTATÍSTICA DO CORPUS ---
frequencia_total = tabela_bow.sum(axis=0) # Soma vertical das ocorrências
top_5 = frequencia_total.nlargest(5)

print("\n📈 PALAVRAS MAIS FREQUENTES (MAIOR PESO NO BoW):")
for palavra, freq in top_5.items():
    print(f"   • '{palavra.ljust(10)}': {int(freq)} ocorrências")

# --- CURIOSIDADE TÉCNICA 3 ---
# O CountVectorizer possui o parâmetro 'stop_words'. Se passarmos a nossa 
# lista customizada da Parte 4 ali, ele já faz a limpeza durante o 'fit'.
# Além disso, o parâmetro 'ngram_range=(1, 2)' permitiria contar duplas de 
# palavras (ex: 'não_recomendo'), resolvendo parte do problema da ordem.
# ----------------------------

print("\n💡 INTERPRETAÇÃO ACADÊMICA:")
print("Note que 'bom' e 'ruim' agora são colunas separadas. O modelo")
print("atribui números às palavras para que o classificador (ML) possa")
print("encontrar padrões geométricos entre as classes Positivo e Negativo.")

📊 ENGENHARIA DE ATRIBUTOS: CONTAGEM DE FREQUÊNCIAS (BoW)

✨ Vocabulário Mapeado (15 palavras únicas):
👉 ['adorei', 'benefício', 'bom', 'compra', 'custo', 'excelente', 'horrível', 'muito', 'não', 'pior', 'produto', 'qualidade', 'recomendo', 'ruim', 'som']

📊 REPRESENTAÇÃO MATRICIAL (VETORES DE FREQUÊNCIA):
                     adorei  benefício  bom  compra  custo  excelente  \
Review 1 (Positivo)       0          0    1       0      0          1   
Review 2 (Negativo)       0          0    0       0      0          0   
Review 3 (Positivo)       0          1    2       0      1          0   
Review 4 (Positivo)       1          0    0       0      0          1   
Review 5 (Negativo)       0          0    0       1      0          0   

                     horrível  muito  não  pior  produto  qualidade  \
Review 1 (Positivo)         0      1    0     0        0          0   
Review 2 (Negativo)         1      0    1     0        1          0   
Review 3 (Positivo)         0      0    0

---
# PARTE 6: TF-IDF (Term Frequency — Inverse Document Frequency)

## ⚖️ O que é TF-IDF?

**Definição:** TF-IDF é uma técnica que mede a **importância real de uma palavra** em um documento, levando em conta o quanto essa palavra é **comum ou rara** no corpus todo.

### Por que importa no pipeline? (Limitação do BoW)

No BoW, **todas as palavras têm o mesmo peso** — a contagem bruta trata "o" e "incrível" como igualmente relevantes. Mas isso é enganoso:

```
Doc 1: "O gato e o gato e o gato"   ← "o" aparece 3x → peso ALTO no BoW
Doc 2: "Machine Learning é incrível" ← "incrível" aparece 1x → peso BAIXO no BoW

❌ PROBLEMA: "o" é super COMUM (aparece em quase todo texto em português!)
✅ TF-IDF: "incrível" ganha peso MUITO maior que "o"
   porque "incrível" é RARO no corpus.
```

### Fórmula: TF-IDF = TF × IDF

| Componente | Nome | O que mede | Efeito no peso |
|-----------|------|-----------|----------------|
| **TF** | Term Frequency | Frequência da palavra **neste** documento | Alta frequência local → TF alto |
| **IDF** | Inverse Document Frequency | Raridade da palavra **no corpus todo** | Palavra rara → IDF alto |

```
Exemplo prático (corpus com 100 documentos):

  Palavra "Python":
    TF  = 3/7 = 0.43  (aparece 3x num doc de 7 palavras)
    IDF = log(100/20) = 1.61  (aparece em 20 dos 100 docs → relativamente rara)
    TF-IDF = 0.43 × 1.61 = 0.69  ← PESO ALTO — palavra importante!

  Palavra "é":
    TF  = 2/7 = 0.29
    IDF = log(100/95) = 0.05  (aparece em 95 dos 100 docs → muito comum)
    TF-IDF = 0.29 × 0.05 = 0.015  ← PESO BAIXO — palavra genérica.
```

> **Resumo em uma frase:** TF-IDF dá peso alto a palavras que são frequentes **neste** documento mas raras **no corpus geral**. Isso destaca automaticamente as palavras mais relevantes de cada texto.

### O que vamos fazer na célula abaixo:
Aplicar o `TfidfVectorizer` no mesmo estilo do `CountVectorizer` e ver como os pesos mudam.


In [ ]:
# =============================================
# TF-IDF COM TfidfVectorizer
# =============================================
# Objetivo: aplicar TF-IDF em avaliações de produtos e
# interpretar quais palavras são realmente importantes.
#
# O TfidfVectorizer funciona igual ao CountVectorizer:
#   fit_transform() → aprende vocabulário + calcula pesos TF-IDF
# A diferença é que os valores NÃO são contagens inteiras,
# mas decimais entre 0 e 1 que representam importância:
#   ~0.0 = palavra pouco relevante (muito comum no corpus)
#   ~1.0 = palavra muito relevante (rara e frequente no doc)

from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

print("=" * 70)
print("DEMONSTRAÇÃO: TF-IDF (TfidfVectorizer)")
print("=" * 70)

corpus_avaliacoes = [
    'Celular excelente, câmera sensacional',   # Doc 1
    'Câmera péssima, celular ruim',            # Doc 2
    'Celular ok, preço bom',                   # Doc 3
]

print("\n📝 Corpus de avaliações:")
for indice, documento in enumerate(corpus_avaliacoes, 1):
    print(f"   {indice}. \"{documento}\"")

# Criar e ajustar o vetorizador TF-IDF
vetorizador_tfidf = TfidfVectorizer(lowercase=True)
matriz_tfidf = vetorizador_tfidf.fit_transform(corpus_avaliacoes)  # aprende + transforma
vocabulario_tfidf = vetorizador_tfidf.get_feature_names_out()

# Visualizar como tabela (arredondando para 4 casas decimais)
tabela_tfidf = pd.DataFrame(
    matriz_tfidf.toarray(),     # esparsa → densa para exibição
    columns=vocabulario_tfidf,
    index=[f"Doc {i+1}" for i in range(len(corpus_avaliacoes))]
)

print("\n📊 Matriz TF-IDF (cada valor = importância da palavra no documento):")
print(tabela_tfidf.round(4))

# Top 3 palavras mais importantes por documento
print("\n🏆 TOP 3 PALAVRAS MAIS RELEVANTES POR DOCUMENTO:")
print("-" * 70)
for indice_doc in range(len(corpus_avaliacoes)):
    pesos = matriz_tfidf[indice_doc].toarray()[0]     # vetor de pesos do doc
    # argsort() retorna índices em ordem crescente; [-3:] pega os 3 maiores;
    # [::-1] inverte para ordem decrescente
    indices_top3 = pesos.argsort()[-3:][::-1]
    print(f"\n📄 Doc {indice_doc + 1}: \"{corpus_avaliacoes[indice_doc]}\"")
    for posicao, idx_palavra in enumerate(indices_top3, 1):
        nome_palavra = vocabulario_tfidf[idx_palavra]
        peso_tfidf = pesos[idx_palavra]
        print(f"   {posicao}. '{nome_palavra}': {peso_tfidf:.4f}")

print("\n💡 Como interpretar:")
print("   • 'sensacional' tem peso ALTO no Doc 1 → só aparece lá (rara no corpus!)")
print("   • 'celular' tem peso BAIXO → aparece em TODOS os docs (comum demais!)")
print("   • TF-IDF destaca automaticamente o que torna cada documento ÚNICO.")

print("\n👉 Próximo passo: comparar BoW e TF-IDF lado a lado para visualizar a diferença.")


---
# PARTE 7: Comparação Prática — BoW vs TF-IDF

## 🔍 Qual a diferença na prática?

Nas partes 5 e 6 vimos BoW e TF-IDF separadamente. Agora vamos usar o **mesmo corpus** e colocar os resultados **lado a lado** para que a diferença fique evidente.

| Aspecto | BoW (`CountVectorizer`) | TF-IDF (`TfidfVectorizer`) |
|---------|------------------------|---------------------------|
| **O que calcula** | Contagem bruta de palavras | Importância relativa (TF × IDF) |
| **Tipo de valor** | Inteiros (0, 1, 2...) | Decimais (0.0 a ~1.0) |
| **Palavras comuns** | Peso alto (aparecem muito) | Peso **baixo** (são genéricas) |
| **Palavras raras** | Peso = 1 (só conta) | Peso **alto** (são únicas!) |
| **Melhor para** | Naive Bayes, tarefas simples | Busca, ranking, clustering, SVM |

> **Regra geral:** Comece com BoW por ser mais simples. Se precisar de resultados melhores, troque por TF-IDF. Ambos alimentam modelos de ML da mesma forma — a diferença está na **qualidade** da representação.


In [ ]:
# ==============================================================================
# BLOCO 5: VETORIZAÇÃO AVANÇADA - MODELO TF-IDF
# Objetivo: Calcular a relevância estatística de cada termo através do produto
# da Frequência do Termo (TF) pelo Inverso da Frequência nos Documentos (IDF).
# ==============================================================================

from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

# --- CURIOSIDADE TÉCNICA 1: O "FILTRO" AUTOMÁTICO ---
# O TF-IDF funciona como um eliminador natural de ruído. Palavras que aparecem
# em praticamente todos os documentos (como 'o', 'que', 'produto') recebem um 
# IDF próximo de zero, perdendo importância na matriz final automaticamente.
# ---------------------------------------------------

print("=" * 70)
print("⚖️ INTELIGÊNCIA ESTATÍSTICA: PESOS TF-IDF")
print("=" * 70)

# Corpus estratégico: Note como a palavra 'celular' se repete em todos.
corpus_avaliacoes = [
    'Celular excelente, câmera sensacional',   # Doc 1
    'Câmera péssima, celular ruim',            # Doc 2
    'Celular ok, preço bom',                   # Doc 3
]

print("\n📝 Amostras do Corpus:")
for indice, documento in enumerate(corpus_avaliacoes, 1):
    print(f"   {indice}. \"{documento}\"")

# Instanciação do Vetorizador TF-IDF
# Curiosidade: Por padrão, o scikit-learn usa a norma 'l2', o que faz com que
# a soma dos quadrados dos pesos de cada linha seja igual a 1.
vetorizador_tfidf = TfidfVectorizer(lowercase=True)

# Processamento: Cálculo dos pesos normalizados
matriz_tfidf = vetorizador_tfidf.fit_transform(corpus_avaliacoes)
vocabulario_tfidf = vetorizador_tfidf.get_feature_names_out()

# --- CURIOSIDADE TÉCNICA 2: A MATEMÁTICA DO LOGARITMO ---
# O cálculo do IDF usa Logaritmo: IDF(t) = log [ n / df(t) ].
# Isso serve para "suavizar" o impacto de palavras muito raras, garantindo
# que uma palavra que aparece em 1 de 1 milhão de docs não tenha um peso 
# absurdamente desproporcional.
# -------------------------------------------------------

# Visualização em DataFrame para análise de pesos
tabela_tfidf = pd.DataFrame(
    matriz_tfidf.toarray(), 
    columns=vocabulario_tfidf, 
    index=[f"Doc {i+1}" for i in range(len(corpus_avaliacoes))]
)

print("\n📊 MATRIZ DE PESOS (RELEVÂNCIA RELATIVA):")
print(tabela_tfidf.round(4))

# --- RANKING DE IMPORTÂNCIA POR DOCUMENTO ---
print("\n🏆 KEYWORDS (PALAVRAS-CHAVE) IDENTIFICADAS:")
print("-" * 70)

for indice_doc in range(len(corpus_avaliacoes)):
    pesos = matriz_tfidf[indice_doc].toarray()[0]
    # Recupera os índices das 3 palavras com maior peso TF-IDF
    indices_top3 = pesos.argsort()[-3:][::-1]
    
    print(f"\n📄 Documento {indice_doc + 1}:")
    for posicao, idx in enumerate(indices_top3, 1):
        palavra = vocabulario_tfidf[idx]
        score = pesos[idx]
        print(f"   {posicao}º '{palavra.ljust(12)}' | Score: {score:.4f}")

# --- CURIOSIDADE TÉCNICA 3: POR QUE NÃO USAR APENAS TF-IDF? ---
# Embora seja superior ao BoW para identificar relevância, o TF-IDF ainda
# ignora o significado. Para o computador, 'sensacional' e 'maravilhoso' 
# continuam sendo apenas números em colunas diferentes, sem relação semântica.
# --------------------------------------------------------------

print("\n💡 INSIGHT PARA A IC:")
print("Observe o termo 'celular'. Embora apareça em todos os documentos,")
print("seu peso é o MENOR da lista. Isso prova que o TF-IDF detectou que")
print("esta palavra não ajuda a diferenciar as avaliações entre si.")

---
# PARTE 8: Classificação de Sentimentos

## 🤖 Fechando o Pipeline: Texto → Números → Modelo → Predição

Até aqui, transformamos texto em números (BoW e TF-IDF). Agora vamos usar esses números como **entrada para modelos de Machine Learning** que classificam reviews como positivas ou negativas.

### Modelos que vamos usar

| Modelo | O que faz | Por que é bom para texto |
|--------|----------|-------------------------|
| **Naive Bayes** (`MultinomialNB`) | Calcula a probabilidade de cada classe usando o Teorema de Bayes | Rápido, simples, funciona surpreendentemente bem com BoW/TF-IDF |
| **Logistic Regression** (`LogisticRegression`) | Encontra uma fronteira linear que separa as classes | Mais robusto com mais dados, excelente baseline |

### Fluxo desta célula:
```
Textos de treino (10 reviews rotuladas)
    ↓ TfidfVectorizer.fit_transform()     ← aprende vocabulário + vetoriza
Matriz TF-IDF de treino
    ↓ modelo.fit()                         ← treina os classificadores
Modelos treinados
    ↓ TfidfVectorizer.transform()          ← vetoriza novas reviews (SEM reaprender)
    ↓ modelo.predict()                     ← classifica
Predições: Positivo ou Negativo
```

> **⚠️ Detalhe importante:** Na hora de vetorizar os dados de teste, usamos `.transform()` (e **não** `.fit_transform()`). Se usássemos `fit_transform`, o vetorizador criaria um **novo vocabulário** baseado nos dados de teste, o que quebraria a compatibilidade com o modelo treinado.

> **Nota:** Nosso dataset tem apenas 10 exemplos de treino — é didático. Em projetos reais, use milhares de exemplos e `train_test_split` para separar treino e teste.


In [13]:
# ==============================================================================
# BLOCO 8: PIPELINE FINAL - CLASSIFICAÇÃO AUTOMÁTICA DE SENTIMENTOS
# Objetivo: Integrar o pré-processamento matemático (TF-IDF) com algoritmos de
# Machine Learning para realizar predições em novos textos.
# ==============================================================================

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
import numpy as np

# --- CURIOSIDADE TÉCNICA 1: GENERALIZAÇÃO ---
# O objetivo de um modelo de IA não é "decorar" o treino, mas aprender padrões.
# Se ele decora, chamamos de Overfitting. Se ele aprende o padrão geral, 
# chamamos de Generalização.
# --------------------------------------------

print("=" * 70)
print("🤖 PIPELINE DE MACHINE LEARNING: TREINAMENTO E INFERÊNCIA")
print("=" * 70)

# ---- PASSO 1: DADOS DE TREINO (Gold Standard) ----
# Conjunto de dados rotulado usado para ajustar os pesos dos modelos.
textos_treino = [
    "Produto excelente, adorei, super recomendo",          # Positivo
    "Muito bom, qualidade incrível, comprem",              # Positivo
    "Ótimo produto, entrega rápida, voltarei a comprar",   # Positivo
    "Gostei muito, superou expectativas",                  # Positivo
    "Produto maravilhoso, melhor compra que fiz",          # Positivo
    "Péssimo produto, não funciona, devolvi",              # Negativo
    "Horrível, quebrou no segundo dia",                    # Negativo
    "Muito ruim, qualidade péssima, não comprem",          # Negativo
    "Produto terrível, parece falsificado",                # Negativo
    "Detestei, pior compra, dinheiro jogado fora",         # Negativo
]
rotulos_treino = [1, 1, 1, 1, 1, 0, 0, 0, 0, 0]  # 1=Positivo, 0=Negativo

# ---- PASSO 2: VETORIZAÇÃO TF-IDF (Aprendizado do Espaço Vetorial) ----
# lowercase=True: Normaliza o case.
# fit_transform: APRENDE o vocabulário do treino e gera a matriz numérica.
vetorizador_classificacao = TfidfVectorizer(lowercase=True)
matriz_treino = vetorizador_classificacao.fit_transform(textos_treino)

print(f"\n📊 Espaço Vetorial de Treino: {matriz_treino.shape[1]} palavras únicas.")

# ---- PASSO 3: TREINAMENTO DOS MODELOS (Fit) ----

# Algoritmo A: Naive Bayes (MultinomialNB)
# Baseado em probabilidades puras. É excelente para textos curtos e datasets 
# pequenos onde as palavras são indicadores fortes.
modelo_naive_bayes = MultinomialNB()
modelo_naive_bayes.fit(matriz_treino, rotulos_treino)

# Algoritmo B: Regressão Logística (Logistic Regression)
# Encontra a melhor linha (hiperplano) que separa os dados no espaço vetorial.
# É mais robusto e lida melhor com relações complexas entre palavras.
modelo_logistic = LogisticRegression(max_iter=1000)
modelo_logistic.fit(matriz_treino, rotulos_treino)

print("✅ Modelos ajustados e prontos para inferência!")

# ---- PASSO 4: TESTE EM DADOS NUNCA VISTOS (Inferência) ----
reviews_teste = [
    "Amei o produto, excelente qualidade",    # Esperado: 1
    "Não gostei, muito ruim",                 # Esperado: 0
    "Bom preço e boa qualidade",              # Esperado: 1
    "Péssimo, nunca mais compro",             # Esperado: 0
    "Produto ok, nada demais",                # Casos de borda (Incerteza)
]

# --- CURIOSIDADE TÉCNICA 2: DATA LEAKAGE (Vazamento de Dados) ---
# Erro comum: Usar .fit_transform nos dados de teste.
# Correto: Usar APENAS .transform(). Os dados de teste DEVEM ser tratados
# seguindo EXATAMENTE o vocabulário aprendido no treino. Se o teste trouxer
# uma palavra nova, ela deve ser ignorada para simular a vida real.
# ----------------------------------------------------------------
matriz_teste = vetorizador_classificacao.transform(reviews_teste)

# Execução das predições
predicoes_nb = modelo_naive_bayes.predict(matriz_teste)
predicoes_lr = modelo_logistic.predict(matriz_teste)

print("\n" + "=" * 70)
print("🔍 RESULTADOS DA CLASSIFICAÇÃO")
print("=" * 70)

for review, pred_nb, pred_lr in zip(reviews_teste, predicoes_nb, predicoes_lr):
    res_nb = "Positivo 😊" if pred_nb == 1 else "Negativo 😞"
    res_lr = "Positivo 😊" if pred_lr == 1 else "Negativo 😞"
    print(f"\n📝 Review: \"{review}\"")
    print(f"   ↳ Predição NB: {res_nb}")
    print(f"   ↳ Predição LR: {res_lr}")

# --- CURIOSIDADE TÉCNICA 3: O "PRODUTO OK" ---
# Note como o modelo lida com "Produto ok, nada demais". Como 'ok' e 'nada' 
# podem não ter peso forte no treino, o modelo recorre à classe estatisticamente
# mais provável ou ao peso de 'produto'. Isso mostra a importância de ter
# dados de treino variados e volumosos.
# --------------------------------------------

print("\n💡 CONCLUSÃO PARA A IC:")
print("A Regressão Logística tende a ser mais 'confiante' em suas separações,")
print("enquanto o Naive Bayes é mais dependente da frequência exata. Em um")
print("projeto real, usaríamos a 'Matriz de Confusão' para medir o erro de cada um.")

🤖 PIPELINE DE MACHINE LEARNING: TREINAMENTO E INFERÊNCIA

📊 Espaço Vetorial de Treino: 42 palavras únicas.
✅ Modelos ajustados e prontos para inferência!

🔍 RESULTADOS DA CLASSIFICAÇÃO

📝 Review: "Amei o produto, excelente qualidade"
   ↳ Predição NB: Positivo 😊
   ↳ Predição LR: Positivo 😊

📝 Review: "Não gostei, muito ruim"
   ↳ Predição NB: Negativo 😞
   ↳ Predição LR: Negativo 😞

📝 Review: "Bom preço e boa qualidade"
   ↳ Predição NB: Positivo 😊
   ↳ Predição LR: Positivo 😊

📝 Review: "Péssimo, nunca mais compro"
   ↳ Predição NB: Negativo 😞
   ↳ Predição LR: Negativo 😞

📝 Review: "Produto ok, nada demais"
   ↳ Predição NB: Positivo 😊
   ↳ Predição LR: Positivo 😊

💡 CONCLUSÃO PARA A IC:
A Regressão Logística tende a ser mais 'confiante' em suas separações,
enquanto o Naive Bayes é mais dependente da frequência exata. Em um
projeto real, usaríamos a 'Matriz de Confusão' para medir o erro de cada um.


---
# PARTE 9: Word Embeddings (Conceitual + Demo)

## 🧠 O que são Word Embeddings?

**Definição:** Word Embeddings são uma forma de representar palavras como **vetores densos de números reais** (geralmente 50 a 300 dimensões), onde palavras com significados parecidos ficam **próximas** no espaço vetorial.

### Por que importa no pipeline?

BoW e TF-IDF são ótimos, mas têm uma limitação fundamental: **não entendem semântica**. Para eles, "bom" e "ótimo" são palavras completamente diferentes (colunas separadas na matriz). Word Embeddings resolvem isso:

```
BoW/TF-IDF: Vetor ESPARSO (muitos zeros, sem semântica)
  Exemplo: [1, 0, 0, 1, 0, 0, 0, ..., 0, 0]   dimensão ~10.000+
  → "bom" e "ótimo" são colunas diferentes, sem relação

Embeddings: Vetor DENSO (valores reais, com semântica)
  Exemplo: [0.25, -0.8, 0.1, ..., 0.9, -0.3]   dimensão 50-300
  → "bom" e "ótimo" têm vetores PRÓXIMOS (significados similares!)
```

### Propriedades Mágicas ✨

```
"gato" ≈ "felino"         → vetores próximos no espaço!
"rei" - "homem" + "mulher" ≈ "rainha"  → aritmética semântica funciona!
```

### Técnicas de Embedding

| Técnica | Origem | Destaque |
|---------|--------|----------|
| **Word2Vec** | Google (2013) | Aprende pelo contexto (Skip-gram / CBOW), rápido |
| **GloVe** | Stanford | Combina frequências globais com contexto local |
| **FastText** | Facebook/Meta | Usa subpalavras (melhor para idiomas com muita morfologia) |
| **BERT/GPT** | Google/OpenAI | Transformers — estado-da-arte atual, contexto dinâmico |

### Quando usar cada representação?

| Situação | Recomendação |
|----------|-------------|
| Classificação simples, poucos dados, interpretabilidade | **BoW ou TF-IDF** |
| Precisa de semântica, dados medianos | **Word2Vec / FastText** |
| Estado-da-arte, tarefas complexas, muitos dados | **BERT / GPT** |

> **Para iniciantes:** BoW e TF-IDF resolvem a grande maioria dos problemas do dia a dia. Embeddings são o próximo nível quando você precisar capturar significado e relações entre palavras.

### O que vamos fazer abaixo:
Treinar um modelo Word2Vec simples com Gensim e explorar similaridades entre palavras.


In [ ]:
# ==============================================================================
# BLOCO 9: WORD EMBEDDINGS - REPRESENTAÇÃO SEMÂNTICA COM WORD2VEC
# Objetivo: Mapear palavras para um espaço vetorial de baixa dimensionalidade,
# onde a proximidade geométrica reflete a similaridade de significado.
# ==============================================================================

import numpy as np

# --- CURIOSIDADE TÉCNICA 1: O FIM DA MATRIZ ESPARSA ---
# Diferente do BoW (que gera milhares de colunas vazias), o Word2Vec gera 
# vetores "DENSOS". Uma palavra não é mais um índice, mas um ponto em um 
# mapa multidimensional (ex: 50 a 300 dimensões).
# -----------------------------------------------------

try:
    from gensim.models import Word2Vec

    print("=" * 70)
    print("🧠 VETORES SEMÂNTICOS: APRENDIZADO DE CONTEXTO (WORD2VEC)")
    print("=" * 70)

    # Dataset Didático: Note que 'gato' e 'cachorro' compartilham verbos, 
    # e 'python' e 'java' compartilham definições.
    sentencas_treino = [
        ['o', 'gato', 'preto', 'dorme'],
        ['o', 'gato', 'branco', 'corre'],
        ['o', 'cachorro', 'preto', 'corre'],
        ['gato', 'e', 'cachorro', 'são', 'animais'],
        ['python', 'é', 'linguagem', 'de', 'programação'],
        ['java', 'é', 'linguagem', 'de', 'programação'],
        ['python', 'é', 'rápido', 'e', 'poderoso'],
    ]

    # --- CURIOSIDADE TÉCNICA 2: HIPERPARÂMETROS ---
    # vector_size: Quantas "características" definem a palavra.
    # window: O tamanho da janela de contexto (quantos vizinhos ele olha).
    # min_count: Ignora palavras muito raras para evitar ruído.
    # ----------------------------------------------
    modelo_w2v = Word2Vec(
        sentences=sentencas_treino,
        vector_size=50,  # Cada palavra vira uma lista de 50 números
        window=3,        # Analisa a vizinhança imediata
        min_count=1,     # Dataset pequeno: não descartamos nada
        workers=1,       # Travado em 1 para garantir o mesmo resultado sempre
        seed=42          # Semente aleatória para reprodutibilidade na IC
    )

    print(f"\n📊 Estatísticas do Modelo:")
    print(f"👉 Vocabulário Ativo: {len(modelo_w2v.wv)} termos")
    print(f"👉 Dimensões por Palavra: {modelo_w2v.vector_size}")

    # Visualização de um vetor real
    palavra_alvo = "gato"
    vetor = modelo_w2v.wv[palavra_alvo] 
    print(f"\n🧠 Representação Vetorial de '{palavra_alvo}' (Primeiras 10/50 dimensões):")
    print(f"   {np.round(vetor[:10], 4)}...")
    
    # --- CURIOSIDADE TÉCNICA 3: SIMILARIDADE DE COSSENO ---
    # Para saber se duas palavras são parecidas, o modelo não subtrai os números,
    # ele calcula o ÂNGULO entre os vetores. Se o ângulo é pequeno, as palavras 
    # são quase sinônimos no contexto do treino.
    # -----------------------------------------------------

    print(f"\n🔗 BUSCA POR VIZINHOS MAIS PRÓXIMOS (Similarity Search):")
    print("-" * 55)
    for consulta in ['gato', 'python', 'programação']:
        if consulta in modelo_w2v.wv:
            # Recupera os termos com maior similaridade de cosseno
            vizinhos = modelo_w2v.wv.most_similar(consulta, topn=3)
            print(f"\n📍 Termo: '{consulta}'")
            for similar, score in vizinhos:
                print(f"   ↳ {similar:<15} | Proximidade: {score:.4%}")

    print("\n💡 INTERPRETAÇÃO PARA A BANCA:")
    print("O modelo aprendeu que 'gato' e 'cachorro' são similares porque")
    print("ambos 'correm' e são 'preto'. Ele não sabe o que é um animal,")
    print("ele sabe que os contextos linguísticos são intercambiáveis.")

except ImportError:
    print("⚠️ Biblioteca Gensim não localizada.")
    print("Dica: !pip install gensim")

print("\n🚀 Pipeline de PLN concluído com sucesso!")

---
# PARTE 10: Conclusão

## 🎯 Resumo do Pipeline PLN

Ao longo deste notebook, percorremos o **pipeline completo** de PLN, desde o texto bruto até a classificação:

```
1. TOKENIZAÇÃO      → Dividir texto em palavras/sentenças (word_tokenize)
2. NORMALIZAÇÃO     → Converter para minúsculas (.lower())
3. STOPWORDS        → Remover palavras genéricas (com cuidado em sentimentos!)
4. VETORIZAÇÃO
   ├── BoW          → Contagem simples de frequências (CountVectorizer)
   ├── TF-IDF       → Pesos inteligentes por raridade (TfidfVectorizer)
   └── Embeddings   → Vetores semânticos densos (Word2Vec)
5. CLASSIFICAÇÃO    → Modelos de ML aplicados (MultinomialNB, LogisticRegression)
```

## 📋 Checklist do que cobrimos

| Tópico | Conceito | Ferramenta Python | Exemplo prático |
|--------|---------|-------------------|----------------|
| Tokenização por palavras | ✅ | `word_tokenize` | ✅ 3 cenários |
| Tokenização por sentenças | ✅ | `sent_tokenize` | ✅ |
| Normalização | ✅ | `.lower()` | ✅ Antes/depois |
| Stopwords (padrão) | ✅ | `nltk.corpus.stopwords` | ✅ 4 reviews |
| Stopwords (customizadas) | ✅ | Operações de `set` | ✅ Preservar negações |
| Bag of Words | ✅ | `CountVectorizer` | ✅ Matriz + top palavras |
| TF-IDF | ✅ | `TfidfVectorizer` | ✅ Matriz + top por doc |
| BoW vs TF-IDF | ✅ | Comparação lado a lado | ✅ Mesmo corpus |
| Classificação (Naive Bayes) | ✅ | `MultinomialNB` | ✅ Treino + predição |
| Classificação (Log. Reg.) | ✅ | `LogisticRegression` | ✅ Treino + predição |
| Word Embeddings | ✅ | `Word2Vec` (Gensim) | ✅ Similaridade |

## 🚀 Próximos Passos

Para continuar aprendendo, veja o notebook **IA_Machine_Learning.ipynb** que contém:
- 🤖 Fundamentos de Inteligência Artificial
- 📈 Modelos de Machine Learning em profundidade
- 🧠 Fundamentos Matemáticos
- ⚡ Limitações e Ética da IA

## 📖 Recursos Recomendados

| Tipo | Recurso |
|------|---------|
| 📚 Livro | *Speech and Language Processing* (Jurafsky & Martin) |
| 🎓 Curso | Natural Language Processing — Coursera / DeepLearning.AI |
| 🐍 Docs | [NLTK](https://www.nltk.org/) · [scikit-learn](https://scikit-learn.org/) |
| 🤗 Comunidade | [Hugging Face](https://huggingface.co/) — transformers modernos |

---

> **Parabéns!** 🎉 Você percorreu o pipeline completo de PLN, desde texto bruto até classificação de sentimentos. Agora tem uma base sólida para explorar projetos mais complexos.
